In [1]:
import sys

In [2]:
sys.getrefcount?

Signature: sys.getrefcount(object, /)
Docstring:
Return the reference count of object.

The count returned is generally one higher than you might expect,
because it includes the (temporary) reference as an argument to
getrefcount().
Type:      builtin_function_or_method

In [3]:
sys.getrefcount("lqkdjqlwdlqh")

3

In [4]:
sys.getrefcount("lqkdjqlwdlqh!")

3

In [5]:
sys.getrefcount([1, 2, 3])

1

In [6]:
sys.getrefcount({1: 11})

1

In [7]:
sys.getrefcount(999)

3

In [8]:
lst = [1, 2, 3]
sys.getrefcount(lst)

2

In [9]:
sys.getrefcount(lst)

2

In [10]:
lst2 = lst
sys.getrefcount(lst)

3

In [11]:
del lst

In [12]:
sys.getrefcount(lst2)

2

In [13]:
def calc_refs(obj):
    local_data = {1: 22}
    print(f"{sys.getrefcount(obj)=}, {sys.getrefcount(lst)=}, {sys.getrefcount(local_data)=}")

In [14]:
lst = [1, 2, 3]
print("before", sys.getrefcount(lst))

calc_refs(lst)

print("after", sys.getrefcount(lst))

before 2
sys.getrefcount(obj)=2, sys.getrefcount(lst)=3, sys.getrefcount(local_data)=1
after 2


In [17]:
sys.getrefcount(1), sys.getrefcount(0), sys.getrefcount(-5), sys.getrefcount(256)

(3221225472, 3221225472, 3221225472, 3221225472)

In [18]:
sys.getrefcount(-6), sys.getrefcount(257)

(2, 3)

In [19]:
val = 1000 - 1000

In [20]:
sys.getrefcount(val)

3221225472

In [21]:
s1 = "qwerty123"
sys.getrefcount(s1)

3

In [22]:
s2 = "qwerty123"
sys.getrefcount(s2)

3

In [23]:
s1 is s2

True

In [24]:
s1 = "qwerty123 "
sys.getrefcount(s1)

3

In [25]:
s2 = "qwerty123 "
sys.getrefcount(s2)

3

In [26]:
s1 is s2

False

In [27]:
s1 = "данные123"
sys.getrefcount(s1)

3

In [28]:
s2 = "данные123"
sys.getrefcount(s2)

3

In [29]:
s1 is s2

False

In [32]:
s1 = "qwerty123"

inner = "qwerty"
s2 = inner + "123"

print(s1 is s2, s1 == s2)

False True


In [36]:
s1 = "qwerty123"

inner = "qwerty"
s2 = sys.intern(inner + "123")

print(s1 is s2, s1 == s2)

True True


In [37]:
s1 = sys.intern("данные123  !!!")

In [38]:
s2 = sys.intern("данные123  !!!")

In [39]:
s1 is s2

True

In [40]:
tp1 = tuple(range(15))

In [41]:
tp1

(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14)

In [42]:
id_tp1 = id(tp1)

In [51]:
id_tp1

4485865088

In [45]:
del tp1

In [46]:
tp21 = tuple(range(14))

In [47]:
id(tp21)

4464173088

In [52]:
tp3 = tuple(range(2, 17))

In [53]:
id(tp3)

4486256720

In [54]:
tp1 = tuple(range(15))
id_tp1 = id(tp1)
print(id_tp1)

4486251616


In [55]:
del tp1

In [56]:
tp1 = tuple(range(15))
id_tp1 = id(tp1)
print(id_tp1)

del tp1

tp2 = tuple(range(14))
print(id(tp2))

tp3 = tuple(range(2, 17))
print(id(tp3), id(tp3) == id_tp1)

4486251616
4470807488
4486251616 True


In [57]:
class Student:
    def __init__(self, name, monitor=None):
        self.name = name
        self.monitor = monitor

    def __str__(self):
        return f"Student[name={self.name}, id={id(self)}]"

    def __del__(self):
        print(f"__del__ Student: {str(self)}")

In [58]:
steve = Student("steve")

In [60]:
print(steve)

Student[name=steve, id=4474956704]


In [61]:
del steve

In [63]:
steve = Student("steve")
print(sys.getrefcount(steve))
del steve

2
__del__ Student: Student[name=steve, id=4475999312]


In [64]:
class Student:
    def __init__(self, name, monitor=None):
        self.name = name
        self.monitor = monitor

    def __str__(self):
        return f"Student[name={self.name}, id={id(self)}]"

    def __del__(self):
        print(f"__del__ Student: {str(self)}")

In [66]:
tom = Student("tom")
alice = Student("alice", tom)
tom.monitor = alice

print(tom, alice)
print(f"111: {sys.getrefcount(tom)=}, {sys.getrefcount(alice)=}")

del tom, alice

Student[name=tom, id=4474966480] Student[name=alice, id=4473214992]
111: sys.getrefcount(tom)=3, sys.getrefcount(alice)=3


In [67]:
import gc

In [68]:
gc.collect()

__del__ Student: Student[name=tom, id=4474952000]
__del__ Student: Student[name=alice, id=4474975120]
__del__ Student: Student[name=tom, id=4474966480]
__del__ Student: Student[name=alice, id=4473214992]


5496

In [69]:
import ctypes

In [78]:
class PyObject(ctypes.Structure):
    _fields_ = [("refcnt", ctypes.c_long)]


def calc_refs(obj_id):
    return PyObject.from_address(obj_id).refcnt


def calc_refs_2(obj_id):
    return ctypes.c_long.from_address(obj_id).value

In [75]:
lst = [1, 2, 3]

PyObject.from_address(id(lst)).refcnt, calc_refs(id(lst))


(1, 1)

In [72]:
PyObject.from_address(id(7)).refcnt

1407378104778752

In [73]:
gc.disable()

In [79]:
tom = Student("tom")
alice = Student("alice", tom)
tom.monitor = alice

id_tom = id(tom)
id_alice = id(alice)

print(tom, alice)
print(f"111: {sys.getrefcount(tom)=}, {sys.getrefcount(alice)=}")
print(f"222a: {calc_refs(id_tom)=}, {calc_refs(id_alice)=}")
print(f"222b: {calc_refs_2(id_tom)=}, {calc_refs_2(id_alice)=}")

del tom, alice

print(f"333a: {calc_refs(id_tom)=}, {calc_refs(id_alice)=}")
print(f"333b: {calc_refs_2(id_tom)=}, {calc_refs_2(id_alice)=}")

gc.collect()

print(f"444a: {calc_refs(id_tom)=}, {calc_refs(id_alice)=}")
print(f"444b: {calc_refs_2(id_tom)=}, {calc_refs_2(id_alice)=}")

Student[name=tom, id=4470050640] Student[name=alice, id=4470048848]
111: sys.getrefcount(tom)=3, sys.getrefcount(alice)=3
222a: calc_refs(id_tom)=2, calc_refs(id_alice)=2
222b: calc_refs_2(id_tom)=2, calc_refs_2(id_alice)=2
333a: calc_refs(id_tom)=1, calc_refs(id_alice)=1
333b: calc_refs_2(id_tom)=1, calc_refs_2(id_alice)=1
__del__ Student: Student[name=tom, id=4470050640]
__del__ Student: Student[name=alice, id=4470048848]
444a: calc_refs(id_tom)=0, calc_refs(id_alice)=0
444b: calc_refs_2(id_tom)=0, calc_refs_2(id_alice)=0


In [80]:
import weakref

In [87]:
tom = Student("tom")

weakref.finalize(tom, lambda *a, **kw: print(f"finalize tom {a=}, {kw=}"))

tom_id = id(tom)
print(f"111: {calc_refs(id(tom))=}")

tom_wr = weakref.ref(tom)
print(f"222: {calc_refs(id(tom))=}")

tom2 = tom_wr()
print(f"333: {calc_refs(id(tom))=}", tom is tom2)

del tom, tom2

print(f"444: {calc_refs(tom_id)=}")

tom = tom_wr()
print(f"{tom=}")

111: calc_refs(id(tom))=1
222: calc_refs(id(tom))=1
333: calc_refs(id(tom))=2 True
__del__ Student: Student[name=tom, id=4463181200]
finalize tom a=(), kw={}
444: calc_refs(tom_id)=1
tom=None


In [88]:
steve = Student("steve")

steve_wr = weakref.proxy(steve)

In [89]:
steve_wr

In [90]:
str(steve_wr)

'Student[name=steve, id=4485821936]'

In [91]:
steve.name, steve.monitor, steve_wr.name, steve_wr.monitor

('steve', None, 'steve', None)

In [92]:
del steve

In [93]:
steve = Student("steve")

steve_wr = weakref.proxy(steve)

print(steve.name, steve.monitor, steve_wr.name, steve_wr.monitor)

del steve

print(steve_wr.name)

steve None steve None
__del__ Student: Student[name=steve, id=4470428048]


ReferenceError: weakly-referenced object no longer exists

In [96]:
sys.getsizeof(1), sys.getsizeof(999), sys.getsizeof(2 ** 100)

(28, 28, 40)

In [99]:
steve = Student("steve")

In [101]:
steve.__dict__

{'name': 'steve', 'monitor': None}

In [102]:
sys.getsizeof([]), sys.getsizeof({}), sys.getsizeof(steve), sys.getsizeof(steve.__dict__)

(56, 64, 48, 184)

In [104]:
sys.getsizeof(()), sys.getsizeof((1,)), sys.getsizeof([1])

(48, 56, 64)

In [106]:
lst = []

for i in range(20):
    print(f"{len(lst)=} {sys.getsizeof(lst)=}")
    lst.append(i)

len(lst)=0 sys.getsizeof(lst)=56
len(lst)=1 sys.getsizeof(lst)=88
len(lst)=2 sys.getsizeof(lst)=88
len(lst)=3 sys.getsizeof(lst)=88
len(lst)=4 sys.getsizeof(lst)=88
len(lst)=5 sys.getsizeof(lst)=120
len(lst)=6 sys.getsizeof(lst)=120
len(lst)=7 sys.getsizeof(lst)=120
len(lst)=8 sys.getsizeof(lst)=120
len(lst)=9 sys.getsizeof(lst)=184
len(lst)=10 sys.getsizeof(lst)=184
len(lst)=11 sys.getsizeof(lst)=184
len(lst)=12 sys.getsizeof(lst)=184
len(lst)=13 sys.getsizeof(lst)=184
len(lst)=14 sys.getsizeof(lst)=184
len(lst)=15 sys.getsizeof(lst)=184
len(lst)=16 sys.getsizeof(lst)=184
len(lst)=17 sys.getsizeof(lst)=248
len(lst)=18 sys.getsizeof(lst)=248
len(lst)=19 sys.getsizeof(lst)=248


In [107]:
dct = {}

for i in range(20):
    print(f"{len(dct)=} {sys.getsizeof(dct)=}")
    dct[i] = i

len(dct)=0 sys.getsizeof(dct)=64
len(dct)=1 sys.getsizeof(dct)=224
len(dct)=2 sys.getsizeof(dct)=224
len(dct)=3 sys.getsizeof(dct)=224
len(dct)=4 sys.getsizeof(dct)=224
len(dct)=5 sys.getsizeof(dct)=224
len(dct)=6 sys.getsizeof(dct)=352
len(dct)=7 sys.getsizeof(dct)=352
len(dct)=8 sys.getsizeof(dct)=352
len(dct)=9 sys.getsizeof(dct)=352
len(dct)=10 sys.getsizeof(dct)=352
len(dct)=11 sys.getsizeof(dct)=632
len(dct)=12 sys.getsizeof(dct)=632
len(dct)=13 sys.getsizeof(dct)=632
len(dct)=14 sys.getsizeof(dct)=632
len(dct)=15 sys.getsizeof(dct)=632
len(dct)=16 sys.getsizeof(dct)=632
len(dct)=17 sys.getsizeof(dct)=632
len(dct)=18 sys.getsizeof(dct)=632
len(dct)=19 sys.getsizeof(dct)=632


In [109]:
tom = Student("tom")

for i in range(20):
    print(f"{len(tom.__dict__)=} {sys.getsizeof(tom.__dict__)=}, {sys.getsizeof(tom)=}")
    setattr(tom, f"attr_{i}", i)

__del__ Student: Student[name=tom, id=4462573904]
len(tom.__dict__)=2 sys.getsizeof(tom.__dict__)=184, sys.getsizeof(tom)=48
len(tom.__dict__)=3 sys.getsizeof(tom.__dict__)=184, sys.getsizeof(tom)=48
len(tom.__dict__)=4 sys.getsizeof(tom.__dict__)=184, sys.getsizeof(tom)=48
len(tom.__dict__)=5 sys.getsizeof(tom.__dict__)=184, sys.getsizeof(tom)=48
len(tom.__dict__)=6 sys.getsizeof(tom.__dict__)=272, sys.getsizeof(tom)=48
len(tom.__dict__)=7 sys.getsizeof(tom.__dict__)=272, sys.getsizeof(tom)=48
len(tom.__dict__)=8 sys.getsizeof(tom.__dict__)=272, sys.getsizeof(tom)=48
len(tom.__dict__)=9 sys.getsizeof(tom.__dict__)=272, sys.getsizeof(tom)=48
len(tom.__dict__)=10 sys.getsizeof(tom.__dict__)=272, sys.getsizeof(tom)=48
len(tom.__dict__)=11 sys.getsizeof(tom.__dict__)=464, sys.getsizeof(tom)=48
len(tom.__dict__)=12 sys.getsizeof(tom.__dict__)=464, sys.getsizeof(tom)=48
len(tom.__dict__)=13 sys.getsizeof(tom.__dict__)=464, sys.getsizeof(tom)=48
len(tom.__dict__)=14 sys.getsizeof(tom.__dict_

In [110]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y


class PointSlots:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y

In [111]:
p1 = Point(10, 20)
print(p1.x, p1.y)

p1.z = 30
print(p1.x, p1.y, p1.z)

10 20
10 20 30


In [113]:
p2 = PointSlots(10, 20)
print(p2.x, p2.y)

p2.x = 99
p2.y = 88

print(p2.x, p2.y)

p2.z = 30
print(p2.x, p2.y, p2.z)

10 20
99 88


AttributeError: 'PointSlots' object has no attribute 'z' and no __dict__ for setting new attributes

In [115]:
dir(p2)

['__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 'x',
 'y']

In [116]:
p1.__dict__

{'x': 10, 'y': 20, 'z': 30}

In [117]:
Point.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 1,
              '__init__': <function __main__.Point.__init__(self, x, y)>,
              '__static_attributes__': ('x', 'y'),
              '__dict__': <attribute '__dict__' of 'Point' objects>,
              '__weakref__': <attribute '__weakref__' of 'Point' objects>,
              '__doc__': None})

In [118]:
PointSlots.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 7,
              '__slots__': ('x', 'y'),
              '__init__': <function __main__.PointSlots.__init__(self, x, y)>,
              '__static_attributes__': ('x', 'y'),
              'x': <member 'x' of 'PointSlots' objects>,
              'y': <member 'y' of 'PointSlots' objects>,
              '__doc__': None})

In [129]:
sys.getsizeof(p2)

48

In [120]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y


class PointSlots(Point):
    __slots__ = ("x", "y")

In [121]:
p3 = PointSlots(10, 20)

In [122]:
p3.x, p3.y

(10, 20)

In [123]:
p3.x = 99

In [124]:
p3.x, p3.y

(99, 20)

In [125]:
p3.z = 60

In [126]:
p3.z

60

In [127]:
p3.__dict__

{'z': 60}

In [128]:
PointSlots.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 7,
              '__slots__': ('x', 'y'),
              '__static_attributes__': (),
              'x': <member 'x' of 'PointSlots' objects>,
              'y': <member 'y' of 'PointSlots' objects>,
              '__doc__': None})

In [130]:
sys.getsizeof(p3)

64

In [131]:
class PointSlotsDict:
    __slots__ = ("x", "y", "__dict__")

    def __init__(self, x, y):
        self.x = x
        self.y = y

p = PointSlotsDict(10, 20)

In [132]:
p.__dict__

{}

In [133]:
p.x, p.y

(10, 20)

In [134]:
p.z = 33

In [135]:
p.__dict__

{'z': 33}